In [1]:
from openai import OpenAI
openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

In [ ]:
from gitsource import chunk_documents

chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)
#step = how much overlap between chunks
#it becomes a sliding window

In [4]:
from minsearch import Index

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

index.fit(chunked_docs)

In [5]:
RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
""".strip()

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [6]:
import json

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(question=question, context=context)

In [7]:
def search(query):
    return index.search(query=query, num_results=5)

In [8]:
def llm(instructions, user_prompt, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    response = openai_client.responses.create(model=model, input=messages)
    return response.output_text

In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(RAG_INSTRUCTIONS, user_prompt)

In [10]:
rag('How do I create a dashboard in Evidently?')

'To create a dashboard in Evidently, you need to follow these steps:\n\n1. **Have Reports in Your Project**: Ensure that you\'ve run an evaluation and saved at least one report in your project. Dashboards rely on reports as their data source.\n\n2. **Add Panels**: You can add panels to your dashboard using the Python API or directly through the UI (Cloud and Enterprise only). Panels can display various visual elements like counters, line plots, and bar plots.\n\n### Using the Python API\n- **Adding a Panel Example**:\n   ```python\n   project.dashboard.add_panel(\n       DashboardPanelPlot(\n           title="Dashboard title",\n           size="full",\n           values=[],  # Specify the metric values\n           plot_params={"plot_type": "text"},\n       ),\n       tab="My new tab",  # Will create a new Tab if it doesn\'t exist\n   )\n   ```\n\n3. **Organizing with Tabs**: You can logically organize panels within different tabs in the dashboard. You can add a new tab by using:\n   ``

In [11]:
from typing import Any, Dict, List

def search(query: str) -> List[Dict[str, Any]]:
    """
    Search the documentation database.

    Args:
        query (str): The search query to look up in the index.

    Returns:
        List of matching documents.
    """
    return index.search(query=query, num_results=5)

In [12]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tool(search)

# Inspect the schema toyaikit generated:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database.\n\nArgs:\n    query (str): The search query to look up in the index.\n\nReturns:\n    List of matching documents.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [14]:
llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=openai_client,
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [15]:
instructions = """
You're a documentation assistant.
Answer the user question using the documentation knowledge base.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.
""".strip()

agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client,
)

In [ ]:
result = agent.loop(
    'How do I create a dahsbord in Evidently?',
    callback=runner_callback
)

print(result.last_message)

-> Response received


-> Response received


To create a dashboard in Evidently, follow these steps:

1. **Prerequisites**:
   - Ensure you have an **Evidently Cloud** account. If you don't, sign up for a [free account](https://app.evidently.cloud/signup).
   - Create an **Organization** after logging in.

2. **Create a Project**:
   - You need to create a Project to associate your dashboard with.

3. **Populate Your Dashboard**:
   - Your dashboard starts empty. You need to run an evaluation and **save at least one Report** in your Project. This will serve as the data source.

4. **Add Panels**:
   - **Via Python API**: You can programmatically add panels using the API. For example, you can use the following code to create a new panel:
     ```python
     from evidently.sdk.panels import DashboardPanelPlot
     from evidently.sdk.models import PanelMetric

     project.dashboard.add_panel(
         DashboardPanelPlot(
             title="Panel Title",
             size="full",
             values=[PanelMetric(legend="Metric Lege

In [ ]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

# Goal is to return snippets
# Indexes entire page into knowledege base, skips chunking

In [19]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}
tokenizer = Tokenizer(stemmer="snowball", stop_words=stopwords)

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=tokenizer,
)

# Returns snippets

In [21]:
query = "how to create a dashboard"
search_results = index.search(query=query, num_results=5)

snippets = highlighter.highlight(query, search_results)
snippets[0]

#Searches whole document, returns a snippet

{'title': 'Overview',
 'description': 'Introduction to Dashboard.',
 'content': {'matches': ['<Check>\n  **Dashboard** is available in **Eviden...',
   '....\n</Check>\n\n## What is a **Dashboard**?\n\nA **Dashboard** provides a...',
   '... What is a **Dashboard**?\n\nA **Dashboard** provides a clear view of...'],
  'total_matches': 18},
 'filename': 'docs/platform/dashboard_overview.mdx'}

In [22]:
file_index = {doc["filename"]: doc["content"] for doc in parsed_docs}

In [24]:
file_index.keys()

dict_keys(['api-reference/endpoint/create.mdx', 'api-reference/endpoint/delete.mdx', 'api-reference/endpoint/get.mdx', 'api-reference/introduction.mdx', 'changelog/changelog.mdx', 'docs/library/data_definition.mdx', 'docs/library/descriptors.mdx', 'docs/library/evaluations_overview.mdx', 'docs/library/leftover_content.mdx', 'docs/library/metric_generator.mdx', 'docs/library/output_formats.mdx', 'docs/library/overview.mdx', 'docs/library/prompt_optimization.mdx', 'docs/library/report.mdx', 'docs/library/synthetic_data_api.mdx', 'docs/library/tags_metadata.mdx', 'docs/library/tests.mdx', 'docs/platform/alerts.mdx', 'docs/platform/dashboard_add_panels.mdx', 'docs/platform/dashboard_add_panels_ui.mdx', 'docs/platform/dashboard_overview.mdx', 'docs/platform/dashboard_panel_types.mdx', 'docs/platform/datasets_generate.mdx', 'docs/platform/datasets_overview.mdx', 'docs/platform/datasets_workflow.mdx', 'docs/platform/evals_api.mdx', 'docs/platform/evals_explore.mdx', 'docs/platform/evals_no_co

In [29]:
from search import SearchTools

In [30]:
search_tools = SearchTools(index, highlighter, file_index)

In [31]:
agent_tools = Tools()
agent_tools.add_tools(search_tools)

In [ ]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2-3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2-3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
""".strip()

agent = OpenAIResponsesRunner(
    tools=
    ,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client,
)

In [39]:
result = agent.run()

-> Response received


-> Response received


Chat ended.


In [36]:
result.__dict__

{'new_messages': [EasyInputMessage(content="You're a documentation assistant.\nAnswer the user question using the documentation knowledge base.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.", role='developer', phase=None, type=None),
  EasyInputMessage(content='How do I create a dahsbord in Evidently?', role='user', phase=None, type=None),
  ResponseFunctionToolCall(arguments='{"query":"create dashboard in Evidently"}', call_id='call_z5JZyhRmdNvrx4X2noWwvJwG', name='search', type='function_call', id='fc_0f6f110ec75288a10069f8784ff5608196beb68e175a856c83', namespace=None, status='completed'),
  {'type': 'function_call_output',
   'call_id': 'call_z5JZyhRmdNvrx4X2noWwvJwG',
   'output': '[\n  {\n    "start": 0,\n    "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI applic

In [41]:
tools = [search_tools.search, search_tools.get_file]

In [42]:
from pydantic_ai import Agent

search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools,
)

In [ ]:
result = await search_agent.run('How do I create a dashboard in Evidently?')
# Uses async.io

In [45]:
print(result.output)

To create a dashboard in Evidently, follow these steps:

1. **Understanding Dashboards**:
   - A Dashboard provides a clear view of your AI application's performance, allowing you to track evaluation results across experiments and monitor production quality over time. Each Project has its own Dashboard, which starts empty until populated with Reports.

2. **Requirements**:
   - You need to **run an evaluation and save at least one Report** to the Project to populate the Dashboard. This Report acts as the data source for the Panels you will create.

3. **Creating a Dashboard**:
    - **Access**: Dashboards are available in **Evidently OSS**, **Evidently Cloud**, and **Evidently Enterprise**.
    - Once you have Reports saved, you can begin organizing your Dashboard.

4. **Adding Panels**:
   - Panels are visual elements in the Dashboard that display specified values in widgets (e.g., counters, line plots, bar plots).
   - You can add Panels in two ways:
     - **Using the Python API**: 

In [46]:
messages = result.all_messages()

In [48]:
for message in messages:
    for part in message.parts:
        print(part)

UserPromptPart(content='How do I create a dashboard in Evidently?', timestamp=datetime.datetime(2026, 5, 4, 11, 32, 25, 269062, tzinfo=datetime.timezone.utc))
ToolCallPart(tool_name='search', args='{"query":"create dashboard in Evidently"}', tool_call_id='call_KptX6v7l8fyjNHTinXH4uI6s')
ToolReturnPart(tool_name='search', content=[{'title': 'Overview', 'description': 'Introduction to Dashboard.', 'content': {'matches': ['<Check>\n  **Dashboard** is available in **Eviden...', '....\n</Check>\n\n## What is a **Dashboard**?\n\nA **Dashboard** provides a...', '... What is a **Dashboard**?\n\nA **Dashboard** provides a clear view of...'], 'total_matches': 18}, 'filename': 'docs/platform/dashboard_overview.mdx'}, {'title': 'Add dashboard panels (API)', 'description': 'How to design your Dashboard with custom Panels.', 'content': {'matches': ['...oard_add_panels_ui).\n\n## **Dashboard** Management\n\n<Check>\n  **Dashboards**...', '...rd Management\n\n<Check>\n  **Dashboards** as code are avai

In [49]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

print_messages(result.all_messages())

request
USER: How do I create a dashboard in Evidently?

response
TOOL CALL: search {"query":"create dashboard in Evidently"}

request
TOOL RETURN: search

response
TOOL CALL: get_file {"filename": "docs/platform/dashboard_overview.mdx"}
TOOL CALL: get_file {"filename": "docs/platform/dashboard_add_panels_ui.mdx"}

request
TOOL RETURN: get_file
TOOL RETURN: get_file

response
To create a dashboard in Evidently, follow these steps:

1. **Understanding Dashboards**:
   - A Dashboard provides a clear view of your AI application's performance, allowing you to track evaluation results across experiments and monitor production quality over time. Each Project has its own Dashboard, which starts empty until populated with Reports.

2. **Requirements**:
   - You need to **run an evaluation and save at least one Report** to the Project to populate the Dashboard. This Report acts as the data source for the Panels you will create.

3. **Creating a Dashboard**:
    - **Access**: Dashboards are avai

In [50]:
messages = result.all_messages()

result2 = await search_agent.run(
    'show me the code',
    message_history=messages
)

print(result2.output)
print_messages(result2.new_messages())

Here's an example code snippet for creating a dashboard and adding panels using the Evidently SDK in Python:

### Setup and Panel Addition Code

```python
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot

# Assuming you have an existing Project
# project = connect_to_your_project()

# Adding a new Tab
project.dashboard.add_tab("My New Tab")

# Adding a Text Panel
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Dashboard Title",
        size="full",
        values=[],  # Leave empty for text panels
        plot_params={"plot_type": "text"},
    ),
    tab="My New Tab"
)

# Adding a Counter Panel for RowCount Metric
# Sum
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Total Row Count",
        subtitle="Total number of evaluations over time.",
        size="half",
        values=[PanelMetric(legend="Row count", metric="RowCount")],
        plot_params={"plot_type": "counter", "aggregation": "sum"},
 

In [51]:
print_messages(result2.all_messages())

request
USER: How do I create a dashboard in Evidently?

response
TOOL CALL: search {"query":"create dashboard in Evidently"}

request
TOOL RETURN: search

response
TOOL CALL: get_file {"filename": "docs/platform/dashboard_overview.mdx"}
TOOL CALL: get_file {"filename": "docs/platform/dashboard_add_panels_ui.mdx"}

request
TOOL RETURN: get_file
TOOL RETURN: get_file

response
To create a dashboard in Evidently, follow these steps:

1. **Understanding Dashboards**:
   - A Dashboard provides a clear view of your AI application's performance, allowing you to track evaluation results across experiments and monitor production quality over time. Each Project has its own Dashboard, which starts empty until populated with Reports.

2. **Requirements**:
   - You need to **run an evaluation and save at least one Report** to the Project to populate the Dashboard. This Report acts as the data source for the Panels you will create.

3. **Creating a Dashboard**:
    - **Access**: Dashboards are avai